# Notebook Template

This template shows how to use the configuration system and reusable modules in Jupyter notebooks.

**No more hardcoded paths!** All data loading uses the centralized configuration.

## Setup

In [ ]:
# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project to path
import sys
sys.path.insert(0, '..')

# Import our utilities
from src.notebook_utils import (
    load_eb_catalog,
    load_panstarrs_data,
    load_ml_data,
    load_model,
    get_config_value,
    get_path,
    save_figure,
    MISSING_VALUE,
    COLOR_THRESHOLD,
    TEST_SIZE,
    RANDOM_STATE
)

from src.features import (
    engineer_all_features,
    select_best_features,
    get_feature_importance
)

# Plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print("✓ Setup complete")
print(f"  Missing value: {MISSING_VALUE}")
print(f"  Test size: {TEST_SIZE}")
print(f"  Random state: {RANDOM_STATE}")

## Load Data

Use the convenience functions to load data - **no paths needed!**

In [ ]:
# Load eclipsing binary catalog
gaia_pm = load_eb_catalog(with_pm=True)
print(f"Gaia catalog: {gaia_pm.shape}")

# Load Pan-STARRS data with temperatures
panstarrs_temp = load_panstarrs_data(with_temps=True)
print(f"Pan-STARRS with temps: {panstarrs_temp.shape}")

# Load ML training data
ml_data = load_ml_data(with_gaia=True)
print(f"ML training data: {ml_data.shape}")

## Feature Engineering

Use the reusable feature engineering functions.

In [ ]:
# Define color columns
color_cols = ['g_r_color', 'r_i_color', 'i_z_color', 'B_V_color', 'bp_rp']
mag_cols = ['gPSFMag']

# Engineer features
df_features = engineer_all_features(
    ml_data,
    color_cols=color_cols,
    mag_cols=mag_cols
)

print(f"Original features: {ml_data.shape[1]}")
print(f"With engineering: {df_features.shape[1]}")
print(f"New features created: {df_features.shape[1] - ml_data.shape[1]}")

## Load a Trained Model

Load models from the models/ directory.

In [ ]:
# Load most recent model
model, metadata = load_model(return_metadata=True)

print("Model loaded:")
print(f"  Type: {type(model).__name__}")
if metadata:
    print(f"  MAE: {metadata.get('mae', 'N/A')}")
    print(f"  R²: {metadata.get('r2', 'N/A')}")

## Plotting and Saving Figures

Use `save_figure()` to automatically save to `reports/figures/`

In [ ]:
# Create a plot
fig, ax = plt.subplots()
ax.hist(panstarrs_temp['Te_avg'].dropna(), bins=50)
ax.set_xlabel('Temperature (K)')
ax.set_ylabel('Count')
ax.set_title('Temperature Distribution')

# Save figure (automatically goes to reports/figures/)
save_figure(fig, 'temperature_distribution.png')

plt.show()

## Access Configuration Values

Get any configuration value using `get_config_value()`

In [ ]:
# Get specific config values
missing_val = get_config_value('processing', 'missing_value')
rf_estimators = get_config_value('ml', 'rf_n_estimators')
gr_coef = get_config_value('temperature', 'gr_coefficients')

print(f"Missing value: {missing_val}")
print(f"RF n_estimators: {rf_estimators}")
print(f"g-r coefficients: {gr_coef}")

## Get Paths

Get any path from configuration.

In [ ]:
# Get paths
data_dir = get_path('processed')
models_dir = get_path('models')
figures_dir = get_path('figures', ensure_exists=True)

print(f"Data directory: {data_dir}")
print(f"Models directory: {models_dir}")
print(f"Figures directory: {figures_dir}")

## Summary

### Benefits of Using These Utilities

✅ **No hardcoded paths** - Everything from configuration
✅ **Portable** - Works on any machine
✅ **Reusable** - Common operations in one place
✅ **Clean** - Less boilerplate code
✅ **Maintainable** - Easy to update

### Available Functions

**Data Loading:**
- `load_eb_catalog()` - Load eclipsing binary catalog
- `load_panstarrs_data()` - Load Pan-STARRS photometry
- `load_ml_data()` - Load ML training data
- `load_colors_temperatures()` - Load colors and temperatures
- `load_model()` - Load trained models

**Feature Engineering:**
- `engineer_all_features()` - Apply all transformations
- `create_polynomial_features()` - Polynomial features
- `create_interaction_features()` - Interaction features
- `create_log_features()` - Log features
- `select_best_features()` - Feature selection
- `get_feature_importance()` - Feature importances

**Utilities:**
- `get_config_value()` - Get configuration values
- `get_path()` - Get paths from config
- `save_figure()` - Save figures automatically

**Constants:**
- `MISSING_VALUE` - Missing value indicator
- `COLOR_THRESHOLD` - Color threshold
- `TEST_SIZE` - Train/test split size
- `RANDOM_STATE` - Random state for reproducibility